# Electrochemical Impedance Spectroscopy (EIS) Analysis
This notebook is designed to process, visualize, and fit EIS data from Biologic (.mpt) files.
It uses `impedance.py` for equivalent circuit fitting and `ipympl` for interactive Matplotlib plots.

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from impedance.models.circuits import CustomCircuit
from impedance.visualization import plot_nyquist, plot_bode
from matplotlib.ticker import AutoMinorLocator

## 1. Data Loading
Robustly read Biologic `.mpt` files. The function skips the metadata headers automatically.

In [19]:
def load_biologic_mpt(filepath):
    # Find the header row
    skip_rows = 0
    with open(filepath, 'r', encoding='latin-1') as f:
        for i, line in enumerate(f):
            if line.startswith('mode'):
                skip_rows = i
                break
                
    # Read the data
    df = pd.read_csv(filepath, sep='\t', skiprows=skip_rows, encoding='latin-1')
    
    if 'time/s' in df.columns:
        # Convert time/s to numeric if possible, otherwise it might be datetime strings
        time_numeric = pd.to_numeric(df['time/s'], errors='coerce')
        if time_numeric.isna().all() and not df['time/s'].isna().all():
            time_dt = pd.to_datetime(df['time/s'], errors='coerce')
            df['time/h'] = (time_dt - time_dt.iloc[0]) / pd.Timedelta(hours=1)
        else:
            df['time/h'] = (time_numeric - time_numeric.iloc[0]) / 3600
    
    if 'z cycle' in df.columns:
        segregation_col = 'z cycle'
    elif 'cycle number' in df.columns and df['cycle number'].nunique() > 1:
        segregation_col = 'cycle number'
    elif 'loop number' in df.columns:
        segregation_col = 'loop number'
    else:
        segregation_col = None
    
    eis_measurements = {}
    metadata = {}
    
    if segregation_col and segregation_col in df.columns:
        for cycle_id, group in df.groupby(segregation_col):
            # Extract state and voltage before EIS
            first_idx = group.index[0]
            if first_idx > 0:
                prev_rows = df.loc[:first_idx-1]
                active_rows = prev_rows[np.abs(prev_rows.get('I/mA', 0)) > 1e-6]
                if not active_rows.empty:
                    last_i = active_rows.iloc[-1]['I/mA']
                    state = "CHARGE" if last_i > 0 else "DISCHARGE"
                else:
                    state = "REST"
                last_ewe = prev_rows.iloc[-1]['Ewe/V'] if 'Ewe/V' in prev_rows.columns else 0.0
            else:
                state = "START"
                last_ewe = group.iloc[0]['Ewe/V'] if 'Ewe/V' in group.columns else 0.0
                
            start_time = group.iloc[0].get('time/h', 0.0)
            start_ewe = group.iloc[0].get('Ewe/V', 0.0)
            
            metadata[int(cycle_id)] = {'state': state, 'voltage': f"{last_ewe:.2f}", 'time': start_time, 'ewe': start_ewe}
            
            # Bulletproof filter: drop empty rows, 0Hz freq, and 0 Ohm impedance points
            group = group.dropna(subset=['freq/Hz'])
            group = group[(group['freq/Hz'] != 0.0) & (group['Re(Z)/Ohm'] != 0.0)] 
            
            # Filter out extreme inductive outliers (negative on the Y-axis) 
            # that ruin the capacitive curve_fit optimizer
            group = group[group['-Im(Z)/Ohm'] > -5.0]
            
            group = group[group['freq/Hz'] >= 0.09]

            if len(group) == 0:
                continue
                
            freq = group['freq/Hz'].values
            Z_real = group['Re(Z)/Ohm'].values
            Z_imag = group['-Im(Z)/Ohm'].values
            Z = Z_real - 1j * Z_imag
            eis_measurements[int(cycle_id)] = (freq, Z)
    else:
        # Fallback if no cycle column exists
        group = df.dropna(subset=['freq/Hz'])
        group = group[(group['freq/Hz'] != 0.0) & (group['Re(Z)/Ohm'] != 0.0)]
        
        freq = group['freq/Hz'].values
        Z_real = group['Re(Z)/Ohm'].values
        Z_imag = group['-Im(Z)/Ohm'].values
        Z = Z_real - 1j * Z_imag
        eis_measurements[1] = (freq, Z)
        metadata[1] = {'state': 'UNKNOWN', 'voltage': 'N/A', 'time': 0, 'ewe': 0}
        
    return eis_measurements, metadata, df

# Provide a dictionary of datasets to load
data_paths = {
    "SI_0": Path("data/20260722_C23_SMSP20_operando_EIS_C01.mpt"),
    "SI_20": Path("data/20260803_3004SI20_C36_1pt19mg_SD3_operandoEIS_C01.mpt"),
    "SI_10": Path("data/20260918_C54_SI10_1pt07mg_operandoEIS_EIS_C01.mpt"),
    "SI_0_New": Path("data/20260923_C57_SMSP20_1pt22mg_operandoEIS.mpt"),
    "SI_20_New": Path("data/20260929_C58_SI20_0pt9mg_operandoEIS_EIS_C01.mpt")
}

eis_dicts = {}
metadata_dicts = {}
raw_dfs = {}
for name, path in data_paths.items():
    if path.exists():
        eis, meta, df = load_biologic_mpt(path)
        eis_dicts[name] = eis
        metadata_dicts[name] = meta
        raw_dfs[name] = df
        print(f"Loaded {name}: Found {len(eis_dicts[name])} EIS measurements.")
    else:
        print(f"Warning: {path} not found.")

Loaded SI_0: Found 11 EIS measurements.
Loaded SI_20: Found 10 EIS measurements.
Loaded SI_10: Found 11 EIS measurements.
Loaded SI_0_New: Found 11 EIS measurements.
Loaded SI_20_New: Found 10 EIS measurements.


# Plot the individual EIS curves

In [20]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from impedance.visualization import plot_nyquist
from IPython.display import display, clear_output

dataset_names = list(eis_dicts.keys())

# Find all common cycle IDs across datasets
all_cycles = set()
for d in eis_dicts.values():
    all_cycles.update(d.keys())
cycle_ids = sorted(list(all_cycles))

def update_plot(cycle_id, selected_datasets):
    # Nature Energy / Publication Quality Settings
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'font.size': 12,          # Scaled up slightly for Jupyter readability 
        'axes.labelsize': 14,     # Journal recommended relative sizing
        'axes.titlesize': 14,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 11,
        'axes.linewidth': 1.5,    # Thicker axes borders
        'lines.linewidth': 2,
        'lines.markersize': 12,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5,
        'xtick.minor.width': 1.0,
        'ytick.minor.width': 1.0,
        
        # Specific theme additions:
        'xtick.direction': 'out',
        'ytick.direction': 'out',
        'xtick.top': False,
        'ytick.right': False,
        'xtick.minor.visible': True,
        'ytick.minor.visible': True,
        'xtick.major.size': 5,      # Smaller ticks
        'ytick.major.size': 5,
        'xtick.minor.size': 3,
        'ytick.minor.size': 3,
        'legend.frameon': False,    # No square box around legend
    })
    
    fig_nyq, ax_nyq = plt.subplots(figsize=(6, 5))
    fig_ec, ax_ec = plt.subplots(figsize=(6, 5))
    
    for ds_name in selected_datasets:
        if cycle_id in eis_dicts[ds_name]:
            freq, Z = eis_dicts[ds_name][cycle_id]
            meta = metadata_dicts[ds_name][cycle_id]
            state = meta['state']
            voltage = meta['voltage']
            
            # Use exact legend format requested by the user
            label = f"{ds_name.upper()}-CYCLE {cycle_id}" #{state} {voltage}V"
            plot_nyquist(Z, ax=ax_nyq, label=label, alpha=0.5)
            
            # Plot EC curve
            df_raw = raw_dfs[ds_name]
            if 'time/h' in df_raw.columns and 'Ewe/V' in df_raw.columns:
                p = ax_ec.plot(df_raw['time/h'], df_raw['Ewe/V'], label=f"{ds_name} EC Curve", alpha=0.8)
                color = p[0].get_color()
                # Red dot highlighting this cycle's location
                ax_ec.plot(meta['time'], meta['ewe'], marker='o', markersize=8, color='red', markeredgecolor='black', markeredgewidth=1.5, zorder=5)
            
    ax_nyq.set_title(f"Operando Nyquist Plot - Cycle {cycle_id}")
    # Explicitly set Nyquist labels to use () instead of []
    ax_nyq.set_xlabel(r"Z' ($\Omega$)")
    ax_nyq.set_ylabel(r"-Z'' ($\Omega$)")
    # Turn off grid for publication style
    ax_nyq.grid(False) 
    ax_nyq.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax_nyq.yaxis.set_minor_locator(AutoMinorLocator(2))
    
    ax_ec.set_title("Electrochemical Profile")
    ax_ec.set_xlabel("Time (h)")
    ax_ec.set_ylabel("Potential (V)")
    # Turn off grid for publication style
    ax_ec.grid(False) 
    ax_ec.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax_ec.yaxis.set_minor_locator(AutoMinorLocator(2))
    
    ax_nyq.set_xlim(0, 1400)
    ax_nyq.set_ylim(0, 1400)
    
    if selected_datasets:
        ax_nyq.legend(loc='best')
        ax_ec.legend(loc='best')
    fig_nyq.tight_layout()
    fig_ec.tight_layout()
    plt.show()
    
    import ipywidgets as widgets
    from IPython.display import display
    
    btn_png = widgets.Button(description='Save as PNG')
    btn_svg = widgets.Button(description='Save as SVG')
    out_msg = widgets.Output()
    
    def save_png(b):
        with out_msg:
            out_msg.clear_output()
            filename = f"operando_nyquist_cycle_{cycle_id}.png"
            fig_nyq.savefig(filename, dpi=300, bbox_inches='tight')
            print(f"Saved {filename} (Nyquist graph only)!")
            
    def save_svg(b):
        with out_msg:
            out_msg.clear_output()
            filename = f"operando_nyquist_cycle_{cycle_id}.svg"
            fig_nyq.savefig(filename, bbox_inches='tight')
            print(f"Saved {filename} (Nyquist graph only)!")
            
    btn_png.on_click(save_png)
    btn_svg.on_click(save_svg)
    
    display(widgets.HBox([btn_png, btn_svg]))
    display(out_msg)

if not cycle_ids:
    print("No data loaded!")
else:
    cycle_slider = widgets.SelectionSlider(
        options=cycle_ids, value=cycle_ids[0], description='Cycle:', continuous_update=False, orientation='horizontal'
    )
    dataset_selector = widgets.SelectMultiple(
        options=dataset_names, value=[dataset_names[0]], description='Datasets:', disabled=False
    )
    widgets.interact(update_plot, cycle_id=cycle_slider, selected_datasets=dataset_selector);

interactive(children=(SelectionSlider(continuous_update=False, description='Cycle:', options=(0, 1, 2, 3, 4, 5…

# Waterfall plot

In [5]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm
import ipywidgets as widgets

def plot_waterfall(selected_datasets):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    colors = ['plasma', 'viridis', 'cividis', 'magma'] 
    
    for i, ds_name in enumerate(selected_datasets):
        ds_cycles = sorted(list(eis_dicts[ds_name].keys()))
        cmap = plt.colormaps.get_cmap(colors[i % len(colors)])
        
        for cycle in ds_cycles:
            freq, Z = eis_dicts[ds_name][cycle]
            Z_re = Z.real
            Z_im_neg = -Z.imag
            y_val = np.full_like(Z_re, cycle)
            
            # Gradient based on cycle number for this specific dataset
            color = cmap(cycle / max(ds_cycles)) if max(ds_cycles) > 0 else 'blue'
            
            # Plot the 3D line
            ax.plot(Z_re, y_val, Z_im_neg, marker='.', markersize=4, linestyle='-', linewidth=1.5, color=color)

    ax.set_xlabel(r"Z' ($\Omega$)", fontsize=12, labelpad=10)
    ax.set_ylabel("Cycle Number", fontsize=12, labelpad=10)
    ax.set_zlabel(r"-Z'' ($\Omega$)", fontsize=12, labelpad=10)
    
    ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.xaxis._axinfo["grid"]['color'] = (0.9, 0.9, 0.9, 1)
    ax.yaxis._axinfo["grid"]['color'] = (0.9, 0.9, 0.9, 1)
    ax.zaxis._axinfo["grid"]['color'] = (0.9, 0.9, 0.9, 1)
    
    ax.view_init(elev=25, azim=-55)
    ax.set_title(f"Operando 3D Waterfall Nyquist Plot ({', '.join(selected_datasets)})", fontsize=14)
    plt.show()

if cycle_ids:
    waterfall_dataset_selector = widgets.SelectMultiple(
        options=dataset_names, value=[dataset_names[0]], description='Datasets:', disabled=False
    )
    widgets.interact(plot_waterfall, selected_datasets=waterfall_dataset_selector);

interactive(children=(SelectMultiple(description='Datasets:', index=(0,), options=('SI_0', 'SI_20', 'SI_10', '…

# 2D plot

In [23]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from impedance.visualization import plot_nyquist

def update_multi_plot(cycle_range, selected_datasets):
    start, end = cycle_range
    fig, ax_nyq = plt.subplots(figsize=(8, 8))
    
    linestyles = ['-', '--', ':', '-.']
    
    for idx, ds_name in enumerate(selected_datasets):
        ds_cycles = sorted(list(eis_dicts[ds_name].keys()))
        selected_cycles = [c for c in ds_cycles if start <= c <= end]
        ls = linestyles[idx % len(linestyles)]
        
        for c in selected_cycles:
            freq, Z = eis_dicts[ds_name][c]
            color = cm.plasma(c / max(ds_cycles)) if max(ds_cycles) > 0 else 'blue'
            
            # Plot the actual data
            plot_nyquist(Z, ax=ax_nyq, fmt=f'.{ls}', color=color)
            
            # Add dummy line to generate a nice legend label
            ax_nyq.plot([], [], f'.{ls}', color=color, label=f'{ds_name} - Cycle {c}')
            
    ax_nyq.set_title(f"Operando Nyquist Plot (Cycles {start} to {end})")
    ax_nyq.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=max(1, len(selected_datasets)//2))
    plt.tight_layout()
    plt.show()

if cycle_ids:
    cycle_range_slider = widgets.IntRangeSlider(
        value=[cycle_ids[0], cycle_ids[-1]],
        min=min(cycle_ids), max=max(cycle_ids), step=1,
        description='Cycles:', continuous_update=False, orientation='horizontal'
    )
    multi_dataset_selector = widgets.SelectMultiple(
        options=dataset_names, value=[dataset_names[0]], description='Datasets:', disabled=False
    )
    widgets.interact(update_multi_plot, cycle_range=cycle_range_slider, selected_datasets=multi_dataset_selector);

interactive(children=(IntRangeSlider(value=(0, 10), continuous_update=False, description='Cycles:', max=10), S…

## 2. Interactive Visualization
Interactive Nyquist and Bode plots. Since `%matplotlib widget` is enabled, you can zoom, pan, and hover over data points.

In [7]:
def plot_eis_data(freq, Z, title="EIS Measurement"):
    fig, (ax_nyq, ax_bode) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Nyquist Plot
    plot_nyquist(Z, ax=ax_nyq)
    ax_nyq.set_title(f"Nyquist Plot - {title}")
    
    # Bode Plot
    plot_bode(freq, Z, axes=ax_bode)
    ax_bode[0].set_title(f"Bode Plot - {title}")
    
    plt.tight_layout()
    plt.show()

# Example usage (plotting the first loop):
# ds_name = list(eis_dicts.keys())[0]
# cycle_id = list(eis_dicts[ds_name].keys())[0]
# freq, Z = eis_dicts[ds_name][cycle_id]
# plot_eis_data(freq, Z, title=f"Operando LiS - {ds_name} Cycle {cycle_id}")

## 3. Equivalent Circuit Modeling & Fitting
Define your equivalent circuit here. 
Since you are working with a LiS battery in a 3-electrode setup, you might start with a Modified Randles circuit, or use multiple RC/RQ elements for the different interfaces (e.g., SEI, charge transfer).

**Common Elements in impedance.py:**
- `R`: Resistor
- `C`: Capacitor
- `CPE`: Constant Phase Element
- `W`: Warburg Element
- `p(R1, C1)`: Parallel elements
- `-`: Series elements

In [ ]:
import pandas as pd
from impedance.models.circuits import CustomCircuit
from IPython.display import display, clear_output
import ipywidgets as widgets

# Define your circuit model
# Example: R0 + p(R1, CPE1) + p(R2, CPE2) + W1
circuit_string = 'R0-p(R1,CPE1)-p(R2,CPE2)-Wo1'
initial_guess = [10, 50, 1e-4, 0.8, 20, 1e-3, 0.8, 100, 10]

#circuit_string = 'R0-p(R1,C1)-W1'
#initial_guess = [10, 50, 1e-4, 100]

dataset_selector_fit = widgets.SelectMultiple(
    options=list(eis_dicts.keys()),
    value=list(eis_dicts.keys()),
    description='Datasets:',
    disabled=False
)

fit_button = widgets.Button(description="Run Fitting (Append)", button_style='success')
clear_button = widgets.Button(description="Clear All Fits", button_style='danger')
fit_output = widgets.Output()

# Initialize globally only if it doesn't exist so we can append across cell re-runs
if 'df_fits' not in globals():
    df_fits = pd.DataFrame()

def on_fit_clicked(b):
    global df_fits
    with fit_output:
        clear_output(wait=True)
        selected_ds = dataset_selector_fit.value
        if not selected_ds:
            print("No datasets selected!")
            return
            
        fit_results = []
        print(f"Fitting model '{circuit_string}' for: {', '.join(selected_ds)}...")
        
        for ds_name in selected_ds:
            # Overwrite previous fits for the SAME dataset if we are re-fitting it
            if not df_fits.empty and 'Dataset' in df_fits.columns:
                df_fits = df_fits[df_fits['Dataset'] != ds_name]
                
            for cycle_id, (freq, Z) in eis_dicts[ds_name].items():
                meta = metadata_dicts[ds_name][cycle_id]
                circuit = CustomCircuit(circuit_string, initial_guess=initial_guess)
                
                try:
                    # Rely on impedance.py's internal bounds to prevent alpha > 1
                    circuit.fit(freq, Z, weight_by_modulus= True)
                    param_names = circuit.get_param_names()[0]
                    param_values = circuit.parameters_
                    param_errors = getattr(circuit, 'conf_', [0.0]*len(param_names))
                    
                    row = {
                        'Dataset': ds_name,
                        'Cycle': cycle_id,
                        'State': meta['state'],
                        'Voltage': float(meta['voltage']),
                        'Time_h': meta['time'],
                        'Circuit': circuit_string,
                        'ParamNames': ','.join(param_names)
                    }
                    
                    for name, val, err in zip(param_names, param_values, param_errors):
                        row[name] = val
                        row[f"{name}_err"] = err
                        
                    fit_results.append(row)
                except Exception as e:
                    print(f"Failed {ds_name} Cycle {cycle_id}: {e}")
                    
        new_fits = pd.DataFrame(fit_results)
        if not df_fits.empty:
            df_fits = pd.concat([df_fits, new_fits], ignore_index=True)
        else:
            df_fits = new_fits
            
        print(f"Fitting complete! Total data points stored: {len(df_fits)}")
        display(df_fits.tail(10))

def on_clear_clicked(b):
    global df_fits
    df_fits = pd.DataFrame()
    with fit_output:
        clear_output()
        print("All stored fits have been cleared.")

fit_button.on_click(on_fit_clicked)
clear_button.on_click(on_clear_clicked)
display(widgets.HBox([dataset_selector_fit, fit_button, clear_button]), fit_output)


Output()

## 3.1 Analyzing Fitted Parameters across SOC/SOD
Track how the fitted parameters (e.g., R0, R1) evolve with cycle number or voltage across your different states.

In [10]:
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

if 'df_fits' not in globals() or df_fits.empty:
    print("No fit data available. Please select datasets and click 'Run Fitting' in the previous cell first.")
else:
    def plot_fitted_params(param_to_plot, x_axis_col, show_errors, exclude_cycles):
        df_plot = df_fits[~df_fits['Cycle'].isin(exclude_cycles)]
        
        if df_plot.empty:
            print("All data points excluded.")
            return
            
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Scatter plot
        sns.scatterplot(
            data=df_plot, 
            x=x_axis_col, 
            y=param_to_plot, 
            hue='Dataset', 
            style='State',
            s=100, 
            ax=ax,
            zorder=3
        )
        
        # Optionally add error bars
        if show_errors and f"{param_to_plot}_err" in df_plot.columns:
            for ds_name in df_plot['Dataset'].unique():
                ds_data = df_plot[df_plot['Dataset'] == ds_name]
                ax.errorbar(
                    ds_data[x_axis_col], ds_data[param_to_plot],
                    yerr=ds_data[f"{param_to_plot}_err"],
                    fmt='none', alpha=0.5, zorder=2, ecolor='black', capsize=3
                )

        # Add connecting lines
        for ds_name in df_plot['Dataset'].unique():
            ds_data = df_plot[df_plot['Dataset'] == ds_name].sort_values(by=x_axis_col)
            ax.plot(ds_data[x_axis_col], ds_data[param_to_plot], alpha=0.3, zorder=1)
            
        ax.set_title(f"Evolution of {param_to_plot} vs {x_axis_col}", fontsize=14)
        ax.grid(True, alpha=0.5)
        plt.tight_layout()
        plt.show()

    # Filter out error columns and metadata for the dropdown
    excluded_cols = ['Dataset', 'Cycle', 'State', 'Voltage', 'Time_h', 'Circuit', 'ParamNames']
    param_options = [col for col in df_fits.columns if col not in excluded_cols and not col.endswith('_err')]
    x_axis_options = ['Cycle', 'Voltage', 'Time_h']
    
    if param_options:
        param_selector = widgets.Dropdown(options=param_options, description='Parameter:')
        x_axis_selector = widgets.Dropdown(options=x_axis_options, description='X-Axis:', value='Cycle')
        error_checkbox = widgets.Checkbox(value=True, description='Show Error Bars')
        
        all_cycles = sorted(df_fits['Cycle'].unique().tolist())
        exclude_selector = widgets.SelectMultiple(
            options=all_cycles, 
            value=[], 
            description='Exclude Cycles:',
            style={'description_width': 'initial'}
        )
        
        # Arrange widgets nicely
        ui = widgets.HBox([
            widgets.VBox([param_selector, x_axis_selector, error_checkbox]), 
            exclude_selector
        ])
        
        out = widgets.interactive_output(plot_fitted_params, {
            'param_to_plot': param_selector, 
            'x_axis_col': x_axis_selector,
            'show_errors': error_checkbox,
            'exclude_cycles': exclude_selector
        })
        display(ui, out)
    else:
        print("No parameters successfully fitted.")


Output()

## 3.2 Visualizing Fit Quality
Compare the raw EIS Nyquist plot against the fitted equivalent circuit model to assess fit quality for a specific dataset and cycle.

In [ ]:
from impedance.visualization import plot_nyquist
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from impedance.models.circuits import CustomCircuit
import numpy as np

if 'df_fits' not in globals() or df_fits.empty:
    print("No fit data available. Please fit your data first.")
else:
    def plot_fit_quality(dataset_name, cycle_id):
        if dataset_name not in eis_dicts or cycle_id not in eis_dicts[dataset_name]:
            print(f"Data for {dataset_name} Cycle {cycle_id} not found.")
            return
            
        freq, Z = eis_dicts[dataset_name][cycle_id]
        
        row = df_fits[(df_fits['Dataset'] == dataset_name) & (df_fits['Cycle'] == cycle_id)]
        if row.empty:
            print(f"No fitting results found for {dataset_name} Cycle {cycle_id}.")
            return
            
        row = row.iloc[0]
        
        c_string = row.get('Circuit', globals().get('circuit_string', 'R0-p(R1,C1)-W1'))
        
        if 'ParamNames' in row and pd.notna(row['ParamNames']):
            param_names = row['ParamNames'].split(',')
        else:
            # Fallback for older fits without ParamNames
            excluded_cols = ['Dataset', 'Cycle', 'State', 'Voltage', 'Time_h', 'Circuit', 'ParamNames']
            param_names = [c for c in df_fits.columns if c not in excluded_cols and not c.endswith('_err')]
            param_names = [p for p in param_names if pd.notna(row.get(p))]
            
        param_values = [row.get(p, 0.0) for p in param_names]
        
        try:
            circuit = CustomCircuit(c_string, initial_guess=param_values)
            circuit.parameters_ = np.array(param_values)
            Z_fit = circuit.predict(freq)
            
            fig, ax = plt.subplots(figsize=(8, 8))
            plot_nyquist(Z, ax=ax, fmt='o', label='Raw Data')
            plot_nyquist(Z_fit, ax=ax, fmt='-', label='Fitted Model')
            
            ax.set_title(f"Fit Quality: {dataset_name} - Cycle {cycle_id}", fontsize=14)
            ax.legend(loc='best')
            plt.tight_layout()
            plt.show()
            
            print("Fitted Parameters:")
            for p, v in zip(param_names, param_values):
                err = row.get(f"{p}_err", 0.0)
                print(f"  {p} = {v:.4e} � {err:.4e}")
                
        except Exception as e:
            print(f"Could not reconstruct circuit for plotting: {e}")

    datasets_with_fits = df_fits['Dataset'].unique().tolist()
    if datasets_with_fits:
        ds_selector = widgets.Dropdown(options=datasets_with_fits, description='Dataset:')
        cycle_selector = widgets.Dropdown(description='Cycle:')
        
        def update_cycles(*args):
            ds = ds_selector.value
            valid_cycles = sorted(df_fits[df_fits['Dataset'] == ds]['Cycle'].unique().tolist())
            cycle_selector.options = valid_cycles
            if valid_cycles:
                cycle_selector.value = valid_cycles[0]
                
        ds_selector.observe(update_cycles, 'value')
        update_cycles() 
        
        widgets.interact(plot_fit_quality, dataset_name=ds_selector, cycle_id=cycle_selector)


interactive(children=(Dropdown(description='Dataset:', options=('SI_0_new', 'SI_20'), value='SI_0_new'), Dropd…

# Overall Electrode Stability (Single file preview)

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def plot_electrode_stability(ds_name):
    if ds_name in raw_dfs:
        df_raw = raw_dfs[ds_name]
        if 'time/h' in df_raw.columns and 'Ece/V' in df_raw.columns:
            fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(10, 7), sharex=True, 
                                           gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.1})
            
            # TOP PLOT
            ax1.plot(df_raw['time/h'], df_raw['Ece/V'], label='Anode vs Ref (Ece)', color='red', linewidth=1.5)
            if 'Ewe/V' in df_raw.columns:
                ax1.plot(df_raw['time/h'], df_raw['Ewe/V'], label='Cathode vs Ref (Ewe)', color='blue', linewidth=1.5)
            ax1.set_ylabel('Potential (V)', fontsize=12)
            ax1.set_title(f'Overall Electrode Stability - {ds_name}', fontsize=14)
            ax1.legend(loc='best')
            ax1.grid(True, alpha=0.5)
            
            # BOTTOM PLOT
            ax2.plot(df_raw['time/h'], df_raw['Ece/V'], label='Anode vs Ref (Ece)', color='red', linewidth=1.5)
            ax2.set_xlabel('Time (Hours)', fontsize=12)
            ax2.set_ylabel('Anode (V)', fontsize=12)
            ax2.legend(loc='best')
            ax2.grid(True, alpha=0.5)
            
            plt.tight_layout()
            plt.show()
        else:
            print(f"Data for {ds_name} does not contain 'time/h' or 'Ece/V'.")
    else:
        print(f"Dataset {ds_name} not found.")

if dataset_names:
    stability_ds_selector = widgets.Dropdown(
        options=dataset_names, value=dataset_names[0], description='Dataset:', disabled=False
    )
    widgets.interact(plot_electrode_stability, ds_name=stability_ds_selector);

interactive(children=(Dropdown(description='Dataset:', options=('SI_0', 'SI_20', 'SI_10', 'SI_0_New', 'SI_20_N…

# DRT analysis (Imaginary part only)

In [ ]:
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
from scipy.optimize import nnls

def compute_drt(freq, Z_imag, lam):
    omega = 2 * np.pi * freq
    tau = 1 / omega
    
    A = np.zeros((len(omega), len(tau)))
    for i, w in enumerate(omega):
        for j, t in enumerate(tau):
            A[i, j] = (w * t) / (1 + (w * t)**2)
            
    n = len(tau)
    L = np.zeros((n-2, n))
    for i in range(n-2):
        L[i, i] = 1
        L[i, i+1] = -2
        L[i, i+2] = 1
        
    A_aug = np.vstack([A, np.sqrt(lam) * L])
    b_aug = np.hstack([Z_imag, np.zeros(n-2)])
    gamma_opt, _ = nnls(A_aug, b_aug)
    return tau, gamma_opt

def update_drt(cycle_id, log_lam, selected_datasets, x_min, x_max, y_min, y_max):
    lam = 10**log_lam
    fig, ax_drt = plt.subplots(figsize=(8, 5))
    
    linestyles = ['-', '--', ':', '-.']
    
    for idx, ds_name in enumerate(selected_datasets):
        if cycle_id in eis_dicts[ds_name]:
            freq, Z = eis_dicts[ds_name][cycle_id]
            Z_imag_data = -Z.imag
            tau, gamma = compute_drt(freq, Z_imag_data, lam)
            
            ls = linestyles[idx % len(linestyles)]
            ax_drt.semilogx(tau, gamma, marker='o', markersize=4, linestyle=ls, label=f'{ds_name}')
    
    ax_drt.set_xlabel(r'Relaxation Time $	au$ (s)', fontsize=12)
    ax_drt.set_ylabel(r'$\gamma(	au)$ ($\Omega$)', fontsize=12)
    ax_drt.set_title(rf"DRT Analysis - Cycle {cycle_id} ($\lambda$ = {lam:.1e})", fontsize=14)
    ax_drt.grid(True, alpha=0.5)
    
    if selected_datasets:
        ax_drt.legend(loc='best')
        
    # Apply x limits before plt.show(). 
    # Pass None if the value is -999 (auto), otherwise use the exact value provided
    left_lim = x_min if x_min != -999 else None
    right_lim = x_max if x_max != -999 else None
    bottom_lim = y_min if y_min != -999 else None
    top_lim = y_max if y_max != -999 else None
    
    ax_drt.set_xlim(left=left_lim, right=right_lim)
    ax_drt.set_ylim(bottom=bottom_lim, top=top_lim)
        
    plt.tight_layout()
    plt.show()

if cycle_ids:
    cycle_slider_drt = widgets.SelectionSlider(
        options=cycle_ids, value=cycle_ids[0], description='Cycle:', continuous_update=False
    )
    lam_slider = widgets.FloatSlider(
        value=-2.0, min=-5.0, max=1.0, step=0.5, 
        description=r'log($\lambda$):', continuous_update=False,
        tooltip="Lower = fits noise (spiky). Higher = over-smoothed."
    )
    drt_dataset_selector = widgets.SelectMultiple(
        options=dataset_names, value=[dataset_names[0]], description='Datasets:', disabled=False
    )
    
    # New text boxes for setting x limits precisely. Default to 0 for auto-scaling.
    x_min_input = widgets.FloatText(value=-999.0, description='X Min (-999=auto):', continuous_update=False)
    x_max_input = widgets.FloatText(value=-999.0, description='X Max (-999=auto):', continuous_update=False)
    y_min_input = widgets.FloatText(value=-999.0, description='Y Min (-999=auto):', continuous_update=False)
    y_max_input = widgets.FloatText(value=-999.0, description='Y Max (-999=auto):', continuous_update=False)

    widgets.interact(update_drt, 
                     cycle_id=cycle_slider_drt, 
                     log_lam=lam_slider, 
                     selected_datasets=drt_dataset_selector,
                     x_min=x_min_input,
                     x_max=x_max_input,
                     y_min=y_min_input,
                     y_max=y_max_input);

# DRT analysis (real+imaginary)

In [ ]:
# ======================================================================
# pyDRTtools-based DRT analysis
#   - Combined Re+Im fit (proper resistance scaling, unlike Im-only)
#   - Extended fine tau grid (better peak resolution)
#   - Inductance handled (high-frequency tail)
#   - GCV (Generalized Cross-Validation) objective lambda selection (defensible, not hand-picked)
#   - Interactive fast path (simple_run) + on-demand Bayesian error bars
#   - Peak quantification: area (= resistance) and FWHM (= homogeneity)
#
# Requires: pip install pyDRTtools cvxopt
# Assumes eis_dicts, metadata_dicts, dataset_names, cycle_ids already defined,
# with eis_dicts[ds][cycle] = (freq_array, Z_complex_array).
# ======================================================================

# ---- robust headless import of pyDRTtools (bypasses its PyQt5 GUI) ----
import matplotlib
matplotlib.use("Agg")  # harmless in notebooks; prevents pyDRTtools' Qt5Agg switch
import sys, types
from unittest.mock import MagicMock
for _n in ['PyQt5', 'PyQt5.QtGui', 'PyQt5.QtWidgets', 'PyQt5.QtCore', 'sip',
           'matplotlib.backends.backend_qt5agg', 'matplotlib.backends.backend_qtagg']:
    sys.modules.setdefault(_n, MagicMock())
sys.modules.setdefault('pyDRTtools.GUI', types.ModuleType('pyDRTtools.GUI'))
sys.modules.setdefault('pyDRTtools.cli', types.ModuleType('pyDRTtools.cli'))
# restore an interactive backend for the actual figures we draw
import matplotlib.pyplot as plt
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass

import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from pyDRTtools import runs as pydrt

# NumPy 2.x renamed trapz -> trapezoid; keep both working
_trapz = getattr(np, "trapezoid", getattr(np, "trapz", None))

# ----------------------------------------------------------------------
# Fitting settings (fixed, defensible defaults)
# ----------------------------------------------------------------------
RBF_TYPE   = 'Gaussian'
DATA_USED  = 'Combined Re-Im Data'   # <-- real + imaginary, the key upgrade
INDUCT     = 1                       # include inductance (HF tail)
DER_ORDER  = '2nd order'             # 2nd-derivative regularization
F_HIGH     = 1e5                     # Hz, drop inductive decade before fitting
F_LOW      = 0.1                     # Hz, drop sub-100 mHz drift

# λ handling: 'GCV (Generalized Cross-Validation)' lets pyDRTtools choose objectively; or supply a fixed value.
CV_TYPE    = 'GCV'


def _prep(freq, Z):
    """Truncate to the trustworthy window and order high->low frequency."""
    freq = np.asarray(freq, float)
    Z = np.asarray(Z, complex)
    m = (freq <= F_HIGH) & (freq >= F_LOW)
    freq, Z = freq[m], Z[m]
    order = np.argsort(freq)[::-1]
    return freq[order], Z[order]


def run_drt(freq, Z, reg_param=None, bayesian=False, nmc=1500):
    """
    Returns dict with tau, gamma (fine grid), residuals, chosen lambda, and
    (if bayesian) lower/upper credible bands.
    reg_param=None -> GCV chooses lambda; else fixed lambda used.
    """
    freq, Z = _prep(freq, Z)
    obj = pydrt.EIS_object(freq, Z.real, Z.imag)
    kw = dict(rbf_type=RBF_TYPE, data_used=DATA_USED, induct_used=INDUCT,
              der_used=DER_ORDER, cv_type=CV_TYPE)
    if reg_param is not None:
        kw['reg_param'] = reg_param
        kw['cv_type'] = 'custom' if 'custom' in _cv_options() else CV_TYPE
    if bayesian:
        obj = pydrt.Bayesian_run(obj, NMC_sample=max(nmc, 600), **kw)
    else:
        obj = pydrt.simple_run(obj, **kw)
    out = {
        'tau':   np.asarray(obj.out_tau_vec),
        'gamma': np.asarray(obj.gamma),
        'lambda': float(np.ravel(obj.lambda_value)[0]),
        'res_re': getattr(obj, 'res_re', None),
        'res_im': getattr(obj, 'res_im', None),
        'freq': freq,
    }
    if bayesian and hasattr(obj, 'lower_bound'):
        out['lower'] = np.asarray(obj.lower_bound)
        out['upper'] = np.asarray(obj.upper_bound)
    return out


def _cv_options():
    return {'GCV', 'mGCV', 'rGCV', 'LC', 're-im', 'kf', 'custom'}


# ----------------------------------------------------------------------
# Peak quantification: integrate area (=resistance) and measure FWHM
# ----------------------------------------------------------------------
def quantify_peaks(tau, gamma, tau_windows):
    """
    For each (tau_lo, tau_hi) window, integrate gamma d(ln tau) -> resistance,
    locate the peak, and estimate FWHM in decades (homogeneity: smaller = more
    uniform / more ideal). Returns list of dicts.
    """
    lnt = np.log(tau)
    results = []
    for (lo, hi) in tau_windows:
        sel = (tau >= lo) & (tau <= hi)
        if sel.sum() < 3:
            results.append({'tau_lo': lo, 'tau_hi': hi, 'R': np.nan,
                            'tau_peak': np.nan, 'fwhm_dec': np.nan})
            continue
        g = gamma[sel]
        x = lnt[sel]
        R = float(_trapz(g, x))                         # area under gamma vs ln(tau) = resistance
        ip = int(np.argmax(g))
        tau_peak = float(tau[sel][ip])
        half = g[ip] / 2.0
        above = x[g >= half]
        fwhm_dec = float((above.max() - above.min()) / np.log(10)) if above.size >= 2 else np.nan
        results.append({'tau_lo': lo, 'tau_hi': hi, 'R': R,
                        'tau_peak': tau_peak, 'fwhm_dec': fwhm_dec})
    return results


# ----------------------------------------------------------------------
# Interactive widget (fast simple_run on the lambda / cycle / dataset sliders)
# ----------------------------------------------------------------------
_LS = ['-', '--', ':', '-.']

def update_drt(cycle_id, log_lam, use_gcv, selected_datasets,
               x_min, x_max, y_min, y_max):
    reg = None if use_gcv else 10.0 ** log_lam
    # Nature Energy / Publication Quality Settings
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'font.size': 12,
        'axes.labelsize': 14,
        'axes.titlesize': 14,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 11,
        'axes.linewidth': 1.5,
        'lines.linewidth': 2,
        'lines.markersize': 0,
        'xtick.direction': 'out',
        'ytick.direction': 'out',
        'xtick.top': False,
        'ytick.right': False,
        'xtick.minor.visible': True,
        'ytick.minor.visible': True,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5,
        'xtick.minor.width': 1.0,
        'ytick.minor.width': 1.0,
        'xtick.major.size': 5,
        'ytick.major.size': 5,
        'xtick.minor.size': 3,
        'ytick.minor.size': 3,
        'legend.frameon': False,
    })
    
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, ds in enumerate(selected_datasets):
        if cycle_id not in eis_dicts[ds]:
            continue
        freq, Z = eis_dicts[ds][cycle_id]
        try:
            r = run_drt(freq, Z, reg_param=reg, bayesian=False)
        except Exception as e:
            print(f"{ds} cycle {cycle_id}: {e}")
            continue
        ax.semilogx(r['tau'], r['gamma'], _LS[i % len(_LS)], marker='o',
                     alpha=0.8, label=f"{ds} (λ={r['lambda']:.1e})")
    ax.set_xlabel(r'Relaxation Time $\tau$ (s)', fontsize=12)
    ax.set_ylabel(r'$\gamma(\tau)$ ($\Omega$)', fontsize=12)
    lam_txt = 'GCV' if use_gcv else f'{10.0**log_lam:.1e}'
    ax.set_title(rf"DRT (pyDRTtools, Re+Im) — Cycle {cycle_id}  |  $\lambda$={lam_txt}", fontsize=13)
    ax.grid(False)
    import matplotlib.ticker as ticker
#    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    if selected_datasets:
        ax.legend(loc='best', fontsize=9)
    ax.set_xlim(left=None if x_min == -999 else x_min, right=None if x_max == -999 else x_max)
    ax.set_ylim(bottom=None if y_min == -999 else y_min, top=None if y_max == -999 else y_max)
    plt.tight_layout()
    plt.show()
    
    import ipywidgets as widgets
    from IPython.display import display
    
    btn_png = widgets.Button(description='Save PNG (600dpi)')
    btn_svg = widgets.Button(description='Save SVG')
    out_msg = widgets.Output()
    
    def save_png(b):
        with out_msg:
            out_msg.clear_output()
            filename = f"drt_cycle_{cycle_id}.png"
            fig.savefig(filename, dpi=600, bbox_inches='tight')
            print(f"Saved {filename} at 600 DPI!")
            
    def save_svg(b):
        with out_msg:
            out_msg.clear_output()
            filename = f"drt_cycle_{cycle_id}.svg"
            fig.savefig(filename, bbox_inches='tight')
            print(f"Saved {filename}!")
            
    btn_png.on_click(save_png)
    btn_svg.on_click(save_svg)
    
    display(widgets.HBox([btn_png, btn_svg]))
    display(out_msg)


if cycle_ids:
    cycle_slider = widgets.SelectionSlider(options=cycle_ids, value=cycle_ids[0],
                                           description='Cycle:', continuous_update=False)
    gcv_toggle   = widgets.Checkbox(value=True, description='Auto λ (GCV)')
    lam_slider   = widgets.FloatSlider(value=-1.5, min=-6.0, max=1.0, step=0.5,
                                       description=r'log(λ):', continuous_update=False)
    ds_selector  = widgets.SelectMultiple(options=dataset_names, value=[dataset_names[0]],
                                          description='Datasets:')
    x_min_in = widgets.FloatText(value=-999.0, description='X min (-999=auto):')
    x_max_in = widgets.FloatText(value=-999.0, description='X max:')
    y_min_in = widgets.FloatText(value=-999.0, description='Y min:')
    y_max_in = widgets.FloatText(value=-999.0, description='Y max:')

    widgets.interact(update_drt,
                     cycle_id=cycle_slider, log_lam=lam_slider, use_gcv=gcv_toggle,
                     selected_datasets=ds_selector,
                     x_min=x_min_in, x_max=x_max_in, y_min=y_min_in, y_max=y_max_in)

interactive(children=(SelectionSlider(continuous_update=False, description='Cycle:', options=(0, 1, 2, 3, 4, 5…

# DRT Multi-Cycle Contour

In [17]:
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.interpolate import interp1d
from IPython.display import display

# We will apply this globally to all datasets
lam = 10**-2.0 

# Pre-compute the DRT curves for all datasets to make the slider fast!
gamma_dict = {}
tau_ref = None

# We use the first dataset to determine tau_ref
if data_paths:
    first_ds = list(data_paths.keys())[0]
    all_cycle_ids = sorted(list(eis_dicts[first_ds].keys()))
    if all_cycle_ids:
        master_freq, master_Z = eis_dicts[first_ds][all_cycle_ids[-1]]
        try:
            master_res = run_drt(master_freq, master_Z, reg_param=lam, bayesian=False)
            tau_ref = master_res['tau']
        except:
            tau_ref = 1 / (2 * np.pi * master_freq)

if tau_ref is not None:
    for ds_name in dataset_names:
        gamma_dict[ds_name] = {}
        for c in eis_dicts[ds_name].keys():
            freq, Z = eis_dicts[ds_name][c]
            try:
                drt_res = run_drt(freq, Z, reg_param=lam, bayesian=False)
                tau, gamma = drt_res['tau'], drt_res['gamma']
            except Exception:
                tau, gamma = compute_drt(freq, -Z.imag, lam)
            f_interp = interp1d(np.log10(tau), gamma, bounds_error=False, fill_value=0.0)
            gamma_dict[ds_name][c] = f_interp(np.log10(tau_ref))
            
    # Find global max for sliders
    global_gamma_max = max([np.max(g) for ds in gamma_dict.values() for g in ds.values()])

    def update_drt_multi(cycle_range, intensity_max, log_tau_max, selected_ds):
        start, end = cycle_range
        # For simplicity, we only contour the FIRST selected dataset
        # Overlapping contours are too confusing.
        ds_name = selected_ds
        ds_cycles = sorted(list(eis_dicts[ds_name].keys()))
        selected_cycles = [c for c in ds_cycles if start <= c <= end]
        
        if len(selected_cycles) < 2:
            print(f"Please select a range of at least 2 cycles for {ds_name} to generate the Contour Map.")
            return
            
        gamma_matrix = np.array([gamma_dict[ds_name][c] for c in selected_cycles])
        
        tau_max = 10**log_tau_max
        tau_min = np.min(tau_ref)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

        # --- PLOT 1: 2D Overlapping Curves ---
        for i, c in enumerate(selected_cycles):
            color = cm.plasma(c / max(ds_cycles)) if max(ds_cycles) > 0 else 'blue'
            ax1.semilogx(tau_ref, gamma_matrix[i], color=color, linewidth=1.5)

        ax1.set_xlim(left=tau_min, right=tau_max)
        ax1.set_ylim(bottom=0, top=intensity_max)
        ax1.set_xlabel('Relaxation Time $\tau$ (s)', fontsize=12)
        ax1.set_ylabel(r'$\gamma(	au)$ ($\Omega$)', fontsize=12)
        ax1.set_title(f'Overlapping DRT ({ds_name} Cycles {start}-{end})', fontsize=14)
        ax1.grid(True, alpha=0.5)

        sm = plt.cm.ScalarMappable(cmap=cm.plasma, norm=plt.Normalize(vmin=min(ds_cycles), vmax=max(ds_cycles)))
        cbar1 = fig.colorbar(sm, ax=ax1)
        cbar1.set_label('Cycle Number')

        # --- PLOT 2: 2D Contour Map (Heatmap) ---
        X, Y = np.meshgrid(tau_ref, selected_cycles)
        contour = ax2.contourf(X, Y, gamma_matrix, levels=np.linspace(0, intensity_max, 30), cmap='viridis', extend='max')

        ax2.set_xscale('log') 
        ax2.set_xlim(left=tau_min, right=0.5)
        ax2.set_xlabel('Relaxation Time $\tau$ (s)', fontsize=12)
        ax2.set_ylabel('Cycle Number', fontsize=12)
        ax2.set_title(f'DRT Contour Map ({ds_name} Cycles {start}-{end})', fontsize=14)

        cbar2 = fig.colorbar(contour, ax=ax2)
        cbar2.set_label(r'$\gamma(	au)$ Intensity ($\Omega$)')

        plt.tight_layout()
        plt.show()

    cycle_range_slider = widgets.IntRangeSlider(
        value=[cycle_ids[0] if cycle_ids else 0, cycle_ids[-1] if cycle_ids else 1],
        min=min(cycle_ids) if cycle_ids else 0, max=max(cycle_ids) if cycle_ids else 1, step=1,
        description='Cycles:', continuous_update=False, orientation='horizontal'
    )

    intensity_slider = widgets.FloatSlider(
        value=global_gamma_max, min=0.1, max=global_gamma_max*1.2, step=0.5,
        description='Intensity:', continuous_update=False, orientation='horizontal',
        tooltip="Lower this to artificially boost the brightness of tiny peaks."
    )

    tau_max_slider = widgets.FloatSlider(
        value=np.log10(np.max(tau_ref)), min=np.log10(np.min(tau_ref)), max=np.log10(np.max(tau_ref)), step=0.1,
        description='log(Max $\tau$):', continuous_update=False, orientation='horizontal',
        tooltip="Slide left to zoom in and hide the high-tau (low frequency) noise."
    )
    
    drt_contour_ds_selector = widgets.Dropdown(
        options=dataset_names, value=dataset_names[0], description='Dataset:', disabled=False
    )

    widgets.interact(update_drt_multi, cycle_range=cycle_range_slider, intensity_max=intensity_slider, log_tau_max=tau_max_slider, selected_ds=drt_contour_ds_selector);

The value of the regularization parameter is 0.01
     pcost       dcost       gap    pres   dres
 0: -3.1247e+06 -3.2061e+06  9e+04  6e+00  1e-04
 1: -3.1387e+06 -3.1530e+06  1e+04  1e+00  2e-05
 2: -3.1399e+06 -3.1408e+06  9e+02  1e-02  2e-07
 3: -3.1399e+06 -3.1400e+06  3e+01  3e-04  5e-09
 4: -3.1399e+06 -3.1399e+06  2e+00  2e-05  3e-10
 5: -3.1399e+06 -3.1399e+06  2e+00  2e-05  3e-10
 6: -3.1399e+06 -3.1400e+06  1e+02  1e-05  2e-10
 7: -3.1399e+06 -3.1399e+06  2e+01  8e-14  2e-16
 8: -3.1399e+06 -3.1399e+06  2e+00  9e-14  5e-19
Optimal solution found.
The value of the regularization parameter is 0.01
     pcost       dcost       gap    pres   dres
 0: -5.8715e+06 -6.1048e+06  2e+05  6e+00  3e-04
 1: -5.9439e+06 -5.9915e+06  5e+04  1e+00  5e-05
 2: -5.9521e+06 -5.9569e+06  5e+03  2e-02  9e-07
 3: -5.9522e+06 -5.9526e+06  4e+02  2e-03  7e-08
 4: -5.9522e+06 -5.9522e+06  3e+01  3e-05  1e-09
 5: -5.9522e+06 -5.9522e+06  1e+01  1e-05  5e-10
 6: -5.9521e+06 -5.9522e+06  8e+01  8e-06  3e

interactive(children=(IntRangeSlider(value=(0, 10), continuous_update=False, description='Cycles:', max=10), F…